In [0]:
-- STEP 0:
-- We will update this record and then time travel to the older version.

SELECT transaction_id, final_amount, updated_at
FROM coffee.silver.transactions
WHERE transaction_id IS NOT NULL
LIMIT 10;


In [0]:
-- STEP 1:
-- This confirms the Silver table is stored in Delta format.
-- Delta format enables ACID transactions + time travel.

DESCRIBE DETAIL coffee.silver.transactions;


In [0]:
-- STEP 2:
-- Delta Lake stores every write (MERGE/UPDATE/DELETE/RESTORE)


DESCRIBE HISTORY coffee.silver.transactions;


In [0]:
-- STEP 3:
-- This shows the current value of final_amount BEFORE the update.

SELECT transaction_id, final_amount, updated_at
FROM coffee.silver.transactions
WHERE transaction_id = '00000f81-7c17-480e-a321-c7fff777b4a5';


In [0]:
-- STEP 4:
-- This UPDATE creates a new Delta version.
-- Delta ensures the update is atomic (all-or-nothing).


UPDATE coffee.silver.transactions
SET final_amount = final_amount + 10
WHERE transaction_id = '00000f81-7c17-480e-a321-c7fff777b4a5';


In [0]:
-- STEP 5:
-- This confirms the update was committed successfully.


SELECT transaction_id, final_amount, updated_at
FROM coffee.silver.transactions
WHERE transaction_id = '00000f81-7c17-480e-a321-c7fff777b4a5';


In [0]:
-- STEP 6:
-- Running history again shows a new version entry for the UPDATE.


DESCRIBE HISTORY coffee.silver.transactions;


In [0]:
-- STEP 7:
-- Time Travel allows reading an older snapshot of the table.


SELECT transaction_id, final_amount, updated_at
FROM coffee.silver.transactions VERSION AS OF 3
WHERE transaction_id = '00000f81-7c17-480e-a321-c7fff777b4a5';


In [0]:
-- STEP 8:
-- It rolls back the table to a previous version.
-- This demonstrates durability + recovery.

RESTORE TABLE coffee.silver.transactions TO VERSION AS OF 3;


In [0]:
-- STEP 9:
-- This confirms the restore succeeded.
-- final_amount should now be back to the original value.

SELECT transaction_id, final_amount, updated_at
FROM coffee.silver.transactions
WHERE transaction_id = '00000f81-7c17-480e-a321-c7fff777b4a5';


In [0]:
-- STEP 10 
-- - UPDATE entry
-- - RESTORE entry
-- Both are stored as Delta versions.

DESCRIBE HISTORY coffee.silver.transactions;
